In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [3]:
# Load data
df = pd.read_csv("credit_card_fraud_case_study_sample.csv")
df = pd.get_dummies(df, drop_first=True)
df.drop_duplicates(inplace=True)

In [4]:
# Features & Target
X = df.drop("isFraud", axis=1)
y = df["isFraud"]

In [5]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
# SMOTE
# Identify rows where y_train is not NaN
not_nan_mask = ~y_train.isna()

# Filter X_train and y_train to remove NaN values from y_train
X_train_cleaned = X_train[not_nan_mask]
y_train_cleaned = y_train[not_nan_mask]

# Apply SMOTE to the cleaned data
X_train, y_train = SMOTE(random_state=42).fit_resample(X_train_cleaned, y_train_cleaned)

In [15]:
# XGBoost
xgb = XGBClassifier(eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [16]:
# Threshold Tuning
prob = xgb.predict_proba(X_test)[:, 1]
pred = (prob > 0.4).astype(int)

print("XGBoost ROC-AUC:", roc_auc_score(y_test, prob))
print(classification_report(y_test, pred))

XGBoost ROC-AUC: 0.44883104362496984
              precision    recall  f1-score   support

         0.0       0.97      1.00      0.98      1383
         1.0       0.00      0.00      0.00        45

    accuracy                           0.97      1428
   macro avg       0.48      0.50      0.49      1428
weighted avg       0.94      0.97      0.95      1428



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
# Top 10 Important Features
imp = pd.Series(xgb.feature_importances_, index=X.columns)
print("\nTop Features:\n", imp.nlargest(10))


Top Features:
 C2                           0.168467
C1                           0.151917
C3                           0.131413
P_emaildomain_yahoo.com      0.082855
ProductCD_S                  0.055332
ProductCD_H                  0.055191
ProductCD_R                  0.046990
P_emaildomain_outlook.com    0.046499
P_emaildomain_hotmail.com    0.040138
ProductCD_W                  0.022478
dtype: float32


In [19]:
# Baseline SVM
svm = SVC(probability=True, random_state=42)
svm.fit(X_train, y_train)

svm_prob = svm.predict_proba(X_test)[:, 1]

print("\nSVM ROC-AUC:", roc_auc_score(y_test, svm_prob))


SVM ROC-AUC: 0.4709729252028601
